In [1]:
# Import shared loading/cleaning functions from utils.py
from utils import load_clean_data
from utils import get_cv_folds

import pandas as pd
import numpy as np

# Models we will train and compare
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import StackingRegressor

# For scaling numeric features (needed for Linear Regression and KNN)
from sklearn.preprocessing import StandardScaler

# For ordinal encoding the categorical columns
from sklearn.preprocessing import OrdinalEncoder

# For evaluation metrics
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

In [2]:
# Load the cleaned dataset
df = load_clean_data("data.csv")

# Generate the FIXED set of cross-validation folds
# Same random_state as every other notebook, so folds match exactly
folds = get_cv_folds(df)

print(f"Loaded {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Generated {len(folds)} folds")

Loaded 79159 rows, 17 columns
Generated 10 folds


In [3]:
# The target variable we're predicting
y = df["advertised_price"]

# Drop the target from the feature set
X = df.drop(columns=["advertised_price"])

# Categorical columns to encode
categorical_cols = ["make", "model", "generation", "fuel", "body",
                     "transmission", "trim", "plate_age_id"]

# Numeric columns — vehicle_age_days EXCLUDED from here onward, since it's
# redundant with vehicle_age_years (confirmed via feature importance in the
# One-Hot notebook: 0.084 vs 0.089), keeping both double-counts the same signal
numeric_cols = ["mileage", "list_price", "reg_year", "doors", "engine",
                "engine_missing", "vehicle_age_years"]

# Sanity check
print(f"Categorical columns: {len(categorical_cols)}")
print(f"Numeric columns: {len(numeric_cols)}")
print(f"Total columns in X: {X.shape[1]}")

Categorical columns: 8
Numeric columns: 7
Total columns in X: 16


In [4]:
# Reset results dictionary — storing RMSE, MAE, and R² for each model, per fold
results = {
    "Linear Regression": {"rmse": [], "mae": [], "r2": []},
    "KNN": {"rmse": [], "mae": [], "r2": []},
    "Decision Tree": {"rmse": [], "mae": [], "r2": []},
    "Random Forest": {"rmse": [], "mae": [], "r2": []},
    "Stacking": {"rmse": [], "mae": [], "r2": []}
}

In [5]:
# Loop through each of the 5 fixed folds
for fold_num, (train_idx, test_idx) in enumerate(folds, start=1):

    # ---- Split data into this fold's train/test portions ----
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # ---- Ordinal Encode the categorical columns ----
    # handle_unknown="use_encoded_value" + unknown_value=-1 means if a category
    # appears in the test fold that was never seen during training, it gets
    # assigned -1 instead of throwing an error (same purpose as One-Hot's
    # handle_unknown="ignore", just Ordinal's version of that safety net)
    oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

    # Fit ONLY on training data for this fold, to avoid leakage
    oe.fit(X_train[categorical_cols])

    # Transform both train and test using the SAME fitted encoder
    X_train_cat = oe.transform(X_train[categorical_cols])
    X_test_cat = oe.transform(X_test[categorical_cols])

    # Convert back into DataFrames with proper column names, so we can
    # concatenate with the numeric columns
    X_train_cat = pd.DataFrame(X_train_cat, columns=categorical_cols, index=X_train.index)
    X_test_cat = pd.DataFrame(X_test_cat, columns=categorical_cols, index=X_test.index)

    # ---- Combine encoded categorical columns with numeric columns ----
    X_train_final = pd.concat([X_train[numeric_cols], X_train_cat], axis=1)
    X_test_final = pd.concat([X_test[numeric_cols], X_test_cat], axis=1)

    # ---- Scale features for models that need it (Linear Regression, KNN) ----
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_final)
    X_test_scaled = scaler.transform(X_test_final)

    # ---- Define the 5 models ----
    linear_model = LinearRegression()
    knn_model = KNeighborsRegressor()
    tree_model = DecisionTreeRegressor(random_state=42)
    rf_model = RandomForestRegressor(random_state=42, n_estimators=50)

    stacking_model = StackingRegressor(
        estimators=[
            ("linear", LinearRegression()),
            ("knn", KNeighborsRegressor()),
            ("tree", DecisionTreeRegressor(random_state=42)),
            ("rf", RandomForestRegressor(random_state=42, n_estimators=50))
        ],
        final_estimator=LinearRegression(),
        cv=3,
    )

    # ---- Train and evaluate each model ----

    # Linear Regression (scaled features)
    linear_model.fit(X_train_scaled, y_train)
    linear_preds = linear_model.predict(X_test_scaled)
    results["Linear Regression"]["rmse"].append(root_mean_squared_error(y_test, linear_preds))
    results["Linear Regression"]["mae"].append(mean_absolute_error(y_test, linear_preds))
    results["Linear Regression"]["r2"].append(r2_score(y_test, linear_preds))

    # KNN (scaled features)
    knn_model.fit(X_train_scaled, y_train)
    knn_preds = knn_model.predict(X_test_scaled)
    results["KNN"]["rmse"].append(root_mean_squared_error(y_test, knn_preds))
    results["KNN"]["mae"].append(mean_absolute_error(y_test, knn_preds))
    results["KNN"]["r2"].append(r2_score(y_test, knn_preds))

    # Decision Tree (unscaled features)
    tree_model.fit(X_train_final, y_train)
    tree_preds = tree_model.predict(X_test_final)
    results["Decision Tree"]["rmse"].append(root_mean_squared_error(y_test, tree_preds))
    results["Decision Tree"]["mae"].append(mean_absolute_error(y_test, tree_preds))
    results["Decision Tree"]["r2"].append(r2_score(y_test, tree_preds))

    # Random Forest (unscaled features)
    rf_model.fit(X_train_final, y_train)
    rf_preds = rf_model.predict(X_test_final)
    results["Random Forest"]["rmse"].append(root_mean_squared_error(y_test, rf_preds))
    results["Random Forest"]["mae"].append(mean_absolute_error(y_test, rf_preds))
    results["Random Forest"]["r2"].append(r2_score(y_test, rf_preds))

    # Stacking (unscaled features)
    stacking_model.fit(X_train_final, y_train)
    stacking_preds = stacking_model.predict(X_test_final)
    results["Stacking"]["rmse"].append(root_mean_squared_error(y_test, stacking_preds))
    results["Stacking"]["mae"].append(mean_absolute_error(y_test, stacking_preds))
    results["Stacking"]["r2"].append(r2_score(y_test, stacking_preds))

    # Print progress so we can see the loop working fold by fold
    print(f"Fold {fold_num}/{len(folds)} complete")

print("\nAll folds complete.")

Fold 1/10 complete
Fold 2/10 complete
Fold 3/10 complete
Fold 4/10 complete
Fold 5/10 complete
Fold 6/10 complete
Fold 7/10 complete
Fold 8/10 complete
Fold 9/10 complete
Fold 10/10 complete

All folds complete.


In [6]:
# Build a summary table: mean and std of RMSE, MAE, and R² across the folds, per model
summary_rows = []
for model_name, metrics in results.items():
    summary_rows.append({
        "model": model_name,
        "mean_rmse": np.mean(metrics["rmse"]),
        "std_rmse": np.std(metrics["rmse"]),
        "mean_mae": np.mean(metrics["mae"]),
        "std_mae": np.std(metrics["mae"]),
        "mean_r2": np.mean(metrics["r2"]),
        "std_r2": np.std(metrics["r2"])
    })

summary = pd.DataFrame(summary_rows)
summary

,model,mean_rmse,std_rmse,mean_mae,std_mae,mean_r2,std_r2
0,Linear Regression,5146.638389,182.354537,3102.530621,42.840780,0.865960,0.006776
1,KNN,2687.427801,142.825047,1551.787734,31.366046,0.963413,0.003323
2,Decision Tree,2937.444105,214.492446,1597.518905,34.716540,0.956205,0.005765
3,Random Forest,2101.879512,135.711501,1154.375711,19.604347,0.977610,0.002449
4,Stacking,2113.241191,132.664400,1175.681281,20.153569,0.977369,0.002402


In [7]:
# Flatten results into a long-format DataFrame: one row per model per fold per metric
# Same structure as the One-Hot notebook, so both files can be combined later
rows = []
for model_name, metrics in results.items():
    for fold_i in range(len(folds)):
        rows.append({
            "encoder": "ordinal",
            "model": model_name,
            "fold": fold_i + 1,
            "rmse": metrics["rmse"][fold_i],
            "mae": metrics["mae"][fold_i],
            "r2": metrics["r2"][fold_i]
        })

results_df = pd.DataFrame(rows)
results_df.to_csv("ordinal_scores.csv", index=False)

print("Saved results to ordinal_scores.csv")

Saved results to ordinal_scores.csv


In [8]:
# Capture feature importance from Random Forest, same as in the One-Hot notebook
feature_importance = pd.DataFrame({
    "feature": X_train_final.columns,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)

feature_importance["encoder"] = "ordinal"
feature_importance.to_csv("ordinal_feature_importance.csv", index=False)

feature_importance.head(15)

,feature,importance,encoder
1,list_price,0.672922,ordinal
6,vehicle_age_years,0.175707,ordinal
0,mileage,0.091893,ordinal
2,reg_year,0.013069,ordinal
8,model,0.010381,ordinal
7,make,0.007853,ordinal
13,trim,0.007141,ordinal
4,engine,0.006727,ordinal
9,generation,0.004972,ordinal
10,fuel,0.002434,ordinal
